# Primera versión: cinco stems a MIDI
Demucs separa `vocals`, `drums`, `bass`, `guitar`, `piano` y `other`. Se entrena un modelo de notas para cada stem excepto `other`; batería usa clases de golpes, los demás notas con altura. Adjuntá [GuitarSet](https://www.kaggle.com/datasets/alonhaviv/guitarset) con **Add Input**. Si no tenés Vocadito y Slakh como entradas, `AUTO_DOWNLOAD_AUX = True` descarga [Vocadito v3](https://zenodo.org/records/5578807) (58 MB) y [BabySlakh](https://zenodo.org/records/4603844) (~883 MB) desde Zenodo. **BabySlakh tiene solo 20 temas: sirve para verificar el pipeline, no para resultados de maestría.** Para entrenar en serio, adjuntá Slakh2100-redux o un subconjunto amplio conservando sus particiones.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
REPO = Path('/kaggle/working/mia-final-project')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Kajachuan/mia-final-project.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

## Entradas y reanudación
Adjuntá la salida anterior como **Notebook Output Files** y completá `PREVIOUS_OUTPUT` para continuar. `TOTAL_EPOCHS` es el objetivo total acumulado. Las descargas auxiliares y las características se omiten si la caché adjunta es compatible.

In [ ]:
GUITARSET_ROOT = Path('/kaggle/input/guitarset')
SLAKH_ROOT = None       # Ej.: '/kaggle/input/slakh2100-redux/slakh2100_flac_redux'
VOCADITO_ROOT = None   # Ej.: '/kaggle/input/vocadito'
PREVIOUS_OUTPUT = None # Ej.: '/kaggle/input/mi-notebook-anterior'
AUTO_DOWNLOAD_AUX = True
REBUILD_AUX_CACHE = False  # True si cambiás de BabySlakh a Slakh2100
MAX_SLAKH_TRACKS = 100  # BabySlakh tiene 20; usar más temas si se adjunta Slakh2100
TOTAL_EPOCHS = 12
RUN_BUDGET_MINUTES = 180
CHECKPOINT_EVERY_BATCHES = 25
WORK = Path('/kaggle/working')
RUNS = WORK / 'runs_five'
RUNS.mkdir(exist_ok=True)
previous = Path(PREVIOUS_OUTPUT) if PREVIOUS_OUTPUT else None
if not GUITARSET_ROOT.is_dir():
    raise FileNotFoundError('Adjuntá alonhaviv/guitarset con Add Input')
if previous and not previous.is_dir():
    raise FileNotFoundError(previous)
STARTED = time.monotonic()

In [ ]:
from polytranscribe.features import FEATURE_VERSION
def compatible(path):
    manifest = path / 'manifest.json'
    return manifest.exists() and json.loads(manifest.read_text()).get('feature_version') == FEATURE_VERSION
GUITAR_CACHE = previous / 'guitarset/cache' if previous and compatible(previous / 'guitarset/cache') else WORK / 'guitarset/cache'
if not compatible(GUITAR_CACHE):
    subprocess.run([sys.executable, '-m', 'polytranscribe.prepare', '--source-root', str(GUITARSET_ROOT),
                    '--root', str(WORK / 'guitarset')], check=True)
    GUITAR_CACHE = WORK / 'guitarset/cache'
AUX_CACHE = previous / 'five_cache' if previous and not REBUILD_AUX_CACHE and all(compatible(previous / 'five_cache' / s) for s in ('vocals','bass','piano','drums')) else WORK / 'five_cache'
if REBUILD_AUX_CACHE or not all(compatible(AUX_CACHE / s) for s in ('vocals','bass','piano','drums')):
    aux = WORK / 'aux'
    if SLAKH_ROOT is None:
        old = previous / 'aux/babyslakh' if previous else None
        if old and (old / '.ready.json').exists():
            SLAKH_ROOT = old
        elif AUTO_DOWNLOAD_AUX:
            from polytranscribe.download_aux import fetch
            SLAKH_ROOT = fetch('babyslakh', aux)
    if VOCADITO_ROOT is None:
        old = previous / 'aux/vocadito' if previous else None
        if old and (old / '.ready.json').exists():
            VOCADITO_ROOT = old
        elif AUTO_DOWNLOAD_AUX:
            from polytranscribe.download_aux import fetch
            VOCADITO_ROOT = fetch('vocadito', aux)
    if SLAKH_ROOT is None or VOCADITO_ROOT is None:
        raise FileNotFoundError('Adjuntá Slakh y Vocadito, o activá AUTO_DOWNLOAD_AUX')
    subprocess.run([sys.executable, '-m', 'polytranscribe.prepare_five',
                    '--slakh-root', str(SLAKH_ROOT), '--vocadito-root', str(VOCADITO_ROOT),
                    '--output', str(WORK / 'five_cache'), '--max-slakh-tracks', str(MAX_SLAKH_TRACKS)], check=True)
    AUX_CACHE = WORK / 'five_cache'
if SLAKH_ROOT is None and previous and (previous / 'aux/babyslakh/.ready.json').exists():
    SLAKH_ROOT = previous / 'aux/babyslakh'
CACHES = {'guitar': GUITAR_CACHE, **{s: AUX_CACHE / s for s in ('vocals','bass','piano','drums')}}
print('Cachés:', CACHES)

## Entrenar y guardar por instrumento
Cada stem tiene checkpoint propio. Se guardan modelo, optimizador y lote cada 25 batches. El presupuesto de tiempo permite parar antes del límite de Kaggle y continuar desde la salida de una versión anterior.

In [ ]:
JOBS = [('guitar','gated'), ('vocals','gated'), ('bass','gated'), ('piano','gated'), ('drums','baseline')]
deadline = STARTED + RUN_BUDGET_MINUTES * 60
for stem, variant in JOBS:
    remaining = (deadline - time.monotonic()) / 60 - 5
    if remaining <= 0:
        print('Presupuesto agotado: guardá esta versión y continuá después')
        break
    output = RUNS / stem
    output.mkdir(exist_ok=True)
    candidates = [output / f'{variant}_last.pt'] if stem == 'guitar' or not REBUILD_AUX_CACHE else []
    if previous and (stem == 'guitar' or not REBUILD_AUX_CACHE):
        candidates.append(previous / 'runs_five' / stem / f'{variant}_last.pt')
    resume = next((path for path in candidates if path.exists()), None)
    command = [sys.executable, '-m', 'polytranscribe.train', '--cache', str(CACHES[stem]),
               '--output', str(output), '--variant', variant, '--epochs', str(TOTAL_EPOCHS),
               '--checkpoint-every', str(CHECKPOINT_EVERY_BATCHES),
               '--max-runtime-minutes', str(remaining)]
    if resume:
        command += ['--resume', str(resume)]
    print('Entrenando', stem, '| resume:', resume, flush=True)
    subprocess.run(command, check=True)
    status = json.loads((output / f'{variant}_status.json').read_text())
    print(stem, status, flush=True)
    if status['status'] != 'complete':
        break

In [ ]:
for stem, variant in JOBS:
    result = RUNS / stem / f'{variant}.json'
    if result.exists():
        scores = json.loads(result.read_text())['test']
        print(stem, 'note/onset F1:', scores['note']['f1'],
              'frame F1:', scores['frame']['f1'])

In [ ]:
MIXTURE_AUDIO = None  # Ejemplo: '/kaggle/input/mi-mezcla/cancion.wav'
ready = all((RUNS / stem / ('baseline.pt' if stem == 'drums' else 'gated.pt')).exists() and
            json.loads((RUNS / stem / ('baseline_status.json' if stem == 'drums' else 'gated_status.json')).read_text())['status'] == 'complete'
            for stem,_ in JOBS)
if ready and time.monotonic() < deadline - 60:
    if MIXTURE_AUDIO:
        audio = Path(MIXTURE_AUDIO)
    elif SLAKH_ROOT:
        mixes = list(Path(SLAKH_ROOT).rglob('mix.wav')) + list(Path(SLAKH_ROOT).rglob('mix.flac'))
        audio = mixes[0]
    else:
        audio = next(GUITARSET_ROOT.rglob('*_mic.wav'))
    subprocess.run([sys.executable, '-m', 'polytranscribe.pipeline', '--audio', str(audio),
                    '--models-root', str(RUNS), '--output-dir', str(RUNS / 'demo')], check=True)
else:
    print('Faltan checkpoints o tiempo para la prueba de cinco stems')

## Interpretación de resultados
Los F1 de GuitarSet, Vocadito y Slakh miden el reconocimiento en stems aislados. Para evaluar mezcla → cinco pistas se necesita un conjunto con referencia de voz además de las cuatro familias de Slakh; BabySlakh no contiene voz. Esta primera versión verifica la salida de cinco pistas pero **no declara una precisión de cinco instrumentos sobre mezclas reales**. Guardá la versión del notebook antes del límite de sesión para persistir `runs_five/` y `five_cache/`.